# LILY'S ABOMINATION™ — Multi-Model AI Video on Kaggle

This launches **WanGP** as one web UI. Inside WanGP's model dropdown you can switch among supported video families instead of rebuilding the notebook.

Good starting points on a Kaggle 16 GB GPU:
- **Wan 2.2 TextImage2Video 5B FastWan** — safest first test, 480p.
- **LTX-2 / LTX-2.3 distilled** — experiment for speed / longer video workflows.
- **HunyuanVideo 1.5** — another image/video option.
- **SkyReels Diffusion Forcing** — legacy long-video experiment if available in the current WanGP build.

Run every cell from top to bottom. Keep the final cell running while you use the public Gradio link.


## 1 — Check the GPU


In [ ]:
import subprocess, sys
subprocess.run(["nvidia-smi"], check=True)

import torch
assert torch.cuda.is_available(), "NO GPU! In Kaggle: Settings → Accelerator → GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)


## 2 — Create storage folders


In [ ]:
from pathlib import Path
import os

ROOT = Path("/kaggle/working/Wan2GP")
DATA = Path("/kaggle/temp/Wan2GP-data")
CKPTS = DATA / "ckpts"
LORAS = DATA / "loras"
CACHE = DATA / "cache"
OUTPUTS = Path("/kaggle/working/Wan2GP-outputs")

for p in (DATA, CKPTS, LORAS, CACHE, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

# Keep enormous downloaded models out of Kaggle's smaller working/output quota.
os.environ["HF_HOME"] = str(CACHE / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE / "huggingface" / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE / "huggingface" / "transformers")
os.environ["TORCH_HOME"] = str(CACHE / "torch")
os.environ["XDG_CACHE_HOME"] = str(CACHE / ".cache")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Models:", CKPTS)
print("Outputs:", OUTPUTS)


## 3 — Download/update WanGP and attach big-data folders


In [ ]:
import subprocess, shutil
from pathlib import Path

REPO = "https://github.com/deepbeepmeep/Wan2GP.git"

if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)

def attach(repo_dir: Path, storage_dir: Path):
    storage_dir.mkdir(parents=True, exist_ok=True)

    if repo_dir.is_symlink():
        # Already correctly attached on a rerun.
        if repo_dir.resolve() == storage_dir.resolve():
            print("Already linked:", repo_dir)
            return
        repo_dir.unlink()

    if repo_dir.exists():
        # Preserve anything already inside before replacing the folder with a link.
        for item in list(repo_dir.iterdir()):
            dest = storage_dir / item.name
            if dest.exists():
                continue
            shutil.move(str(item), str(dest))
        shutil.rmtree(repo_dir)

    repo_dir.symlink_to(storage_dir, target_is_directory=True)
    print("Linked:", repo_dir, "->", storage_dir)

attach(ROOT / "ckpts", CKPTS)
attach(ROOT / "loras", LORAS)
attach(ROOT / "outputs", OUTPUTS)


## 4 — Install video/system libraries


In [ ]:
import subprocess, os, shutil

env = os.environ.copy()
env["DEBIAN_FRONTEND"] = "noninteractive"

prefix = [] if os.geteuid() == 0 else ["sudo"]
subprocess.run(prefix + ["apt-get", "update", "-qq"], check=True, env=env)
subprocess.run(
    prefix + [
        "apt-get", "install", "-y", "--no-install-recommends",
        "ffmpeg", "libglib2.0-0", "libgl1", "libportaudio2"
    ],
    check=True,
    env=env
)
print("System libraries installed.")


## 5 — Install WanGP WITHOUT replacing Kaggle's working CUDA Torch

This deliberately pins the Torch/Torchvision/Torchaudio versions that Kaggle already booted with.


In [ ]:
import importlib, subprocess, sys, os
from pathlib import Path
import torch

assert torch.cuda.is_available()

installed = {"torch": torch.__version__}
for name in ("torchvision", "torchaudio"):
    try:
        installed[name] = importlib.import_module(name).__version__
    except Exception:
        installed[name] = None

constraints = CACHE / "kaggle-torch-constraints.txt"
lines = [f"torch=={torch.__version__.split('+', 1)[0]}"]

for name in ("torchvision", "torchaudio"):
    ver = installed.get(name)
    if ver:
        lines.append(f"{name}=={ver.split('+', 1)[0]}")

constraints.write_text("\n".join(lines) + "\n")
print(constraints.read_text())

env = os.environ.copy()
env["PIP_NO_CACHE_DIR"] = "1"

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-cache-dir", "--upgrade", "setuptools", "wheel"],
    check=True, env=env
)

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-cache-dir",
     "--upgrade-strategy", "only-if-needed",
     "-r", str(ROOT / "requirements.txt"),
     "-c", str(constraints)],
    check=True, env=env
)

print("WanGP requirements installed.")
print("GPU still alive:", torch.cuda.get_device_name(0))


## 6 — Kaggle headless-display fix


In [ ]:
target = ROOT / "preprocessing/matanyone/tools/interact_tools.py"

if target.exists():
    txt = target.read_text()
    txt2 = txt.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
    if txt2 != txt:
        target.write_text(txt2)
        print("Patched TkAgg -> Agg.")
    else:
        print("No matplotlib patch needed.")
else:
    print("Patch target not present; skipping.")


## 7 — LAUNCH THE BEAST

Wait until the output prints a `gradio.live` public link, then tap it on your iPhone.

**Do not stop this cell while using the UI.**

Inside WanGP, use the model dropdown. On a T4, begin modestly (480p / short clip) before increasing duration or resolution.


In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env["WAN_CACHE_DIR"] = str(CACHE)
env["HF_HOME"] = str(CACHE / "huggingface")
env["HUGGINGFACE_HUB_CACHE"] = str(CACHE / "huggingface" / "hub")
env["TRANSFORMERS_CACHE"] = str(CACHE / "huggingface" / "transformers")
env["TORCH_HOME"] = str(CACHE / "torch")
env["XDG_CACHE_HOME"] = str(CACHE / ".cache")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    sys.executable, "-u", "wgp.py",
    "--listen",
    "--server-port", "7860",
    "--share",
    "--profile", "5",
]

print("Launching WanGP...")
print("When you see the PUBLIC gradio.live URL, TAP IT.")
print()

p = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def heartbeat():
    while p.poll() is None:
        time.sleep(45)
        if p.poll() is None:
            print("[still running — keep this cell open]")

threading.Thread(target=heartbeat, daemon=True).start()

try:
    for line in iter(p.stdout.readline, ""):
        if not line:
            break
        print(line, end="")
except KeyboardInterrupt:
    p.terminate()
    print("WanGP stopped.")
